In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torchaudio
import torch
from torch.utils.data import Dataset, DataLoader
from IPython.display import Audio

In [3]:
# Пути к CSV-файлам
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

In [4]:
# class MorseDataset(Dataset):
#     def __init__(self, df, audio_dir, sample_rate=48000, n_mels=64, transform=None):
#         self.df = df
#         self.audio_dir = audio_dir
#         self.sample_rate = sample_rate
#         self.n_mels = n_mels
#         self.transform = transform
# 
#         # Преобразование аудио в мел-спектрограмму
#         self.mel_extractor = torchaudio.transforms.MelSpectrogram(
#             sample_rate=self.sample_rate,
#             n_fft=1024,
#             hop_length=512,
#             n_mels=self.n_mels
#         )
#         self.db_transform = torchaudio.transforms.AmplitudeToDB()
# 
#     def __len__(self):
#         return len(self.df)
# 
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         filename = row["id"]
#         filepath = os.path.join(self.audio_dir, filename)
# 
#         # Загрузка аудио
#         waveform, sr = torchaudio.load(filepath)
# 
#         # Преобразование в Mel Spectrogram
#         mel_spec = self.mel_extractor(waveform)
#         mel_spec_db = self.db_transform(mel_spec)
# 
#         # Нормализация (опционально — но помогает модели)
#         mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-6)
# 
#         # Выход: [1, n_mels, time]
#         mel_spec_db = mel_spec_db.squeeze(0)  # -> [n_mels, time]
# 
#         # Возвращаем: признаки + целевой текст
#         return mel_spec_db, row["message"]

In [114]:
import librosa

AUDIO_DIR = "morse_dataset"

filename = train_df.iloc[0]["id"]  # например, "1.opus"
filepath = os.path.join(AUDIO_DIR, filename)
print(filepath)

waveform, sr = librosa.load(filepath, sr=16000)  # sr можно выбрать


morse_dataset\1.opus


In [115]:
print("waveform shape:", waveform.shape)
print("duration (сек):", len(waveform) / sr)


waveform shape: (128000,)
duration (сек): 8.0


In [116]:
print("Мин/Макс:", waveform.min(), waveform.max())
print("Средняя амплитуда:", np.mean(np.abs(waveform)))


Мин/Макс: -1.3791139 1.333097
Средняя амплитуда: 0.5653127


In [1]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import os

def extract_log_mel_spectrogram(audio_path, sr=8000, n_mels=32, hop_length=256, n_fft=512):
    # Загружаем аудио
    y, sr = librosa.load(audio_path, sr=sr)

    # Извлекаем мел-спектрограмму
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)

    # Преобразуем в логарифмическую шкалу
    log_mel_spec = librosa.power_to_db(mel_spec, ref=np.max)

    return log_mel_spec



In [2]:

# Пример использования:
audio_file = filepath
features = extract_log_mel_spectrogram(audio_file)

# Визуализация (по желанию)
plt.figure(figsize=(10, 4))
librosa.display.specshow(features, sr=16000, hop_length=256, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Log-Mel Spectrogram')
plt.tight_layout()
plt.show()

NameError: name 'filepath' is not defined

In [74]:
train_df

,id,message
0,1.opus,03ЩУЫЛПИГХ
1,2.opus,ЪЛТ0ДС6А3Г
2,3.opus,5ЭКЫБЗХЯН
3,4.opus,ЖЫЦОИ68КФ
4,5.opus,32Ю7МЫ ЗЛ
...,...,...
29995,29996.opus,ЬДТРЭ 9М6М
29996,29997.opus,ЬКТ1 ШЭЪ
29997,29998.opus,ЫВЙЗБЯН7К
29998,29999.opus,ФЯДШ3Т#


In [119]:
from torch.utils.data import Dataset
import torch

class MorseDataset(Dataset):
    def __init__(self, dataframe, audio_dir, sr=16000, transform=None):
        self.data = dataframe
        self.audio_dir = audio_dir
        self.sr = sr
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row["id"])  # имя файла в csv
        label = row["message"]  # допустим, текстовая расшифровка

        # извлекаем спектрограмму
        spec = extract_log_mel_spectrogram(audio_path, sr=self.sr)

        if self.transform:
            spec = self.transform(spec)

        # Преобразуем в тензор PyTorch
        spec = torch.tensor(spec).float()
        label = str(label)  # оставим как текст пока

        return spec, label


In [120]:
dataset = MorseDataset(dataframe=train_df, audio_dir="morse_dataset")
print("Размер датасета:", len(dataset))

# Проверим 1 пример
spec, label = dataset[29999]
print("📐 Спектрограмма:", spec.shape)
print("🔤 Метка:", label)


Размер датасета: 30000
📐 Спектрограмма: torch.Size([32, 251])
🔤 Метка: ЪЭ8Д2МЗА


In [121]:
# Создаём словарь
all_texts = "".join(train_df["message"].values)  # или train_df
unique_chars = sorted(set("".join(all_texts)))
char2idx = {ch: i+1 for i, ch in enumerate(unique_chars)}
char2idx[0] = '<pad>'

# Обратный словарь
idx2char = {i: ch for ch, i in char2idx.items()}

# Функция кодирования строки
def text_to_indices(text):
    return [char2idx[ch] for ch in text]

In [112]:
char2idx

{' ': 1,
 '#': 2,
 '0': 3,
 '1': 4,
 '2': 5,
 '3': 6,
 '4': 7,
 '5': 8,
 '6': 9,
 '7': 10,
 '8': 11,
 '9': 12,
 'А': 13,
 'Б': 14,
 'В': 15,
 'Г': 16,
 'Д': 17,
 'Е': 18,
 'Ж': 19,
 'З': 20,
 'И': 21,
 'Й': 22,
 'К': 23,
 'Л': 24,
 'М': 25,
 'Н': 26,
 'О': 27,
 'П': 28,
 'Р': 29,
 'С': 30,
 'Т': 31,
 'У': 32,
 'Ф': 33,
 'Х': 34,
 'Ц': 35,
 'Ч': 36,
 'Ш': 37,
 'Щ': 38,
 'Ъ': 39,
 'Ы': 40,
 'Ь': 41,
 'Э': 42,
 'Ю': 43,
 'Я': 44,
 0: '<pad>'}

In [78]:
# # Собираем алфавит из всех символов train
# all_text = "".join(train_df["message"].values)
# vocab = sorted(set(all_text))
# 
# # Создаем маппинг символов в индексы
# char2idx = {c: i + 1 for i, c in enumerate(vocab)}  # 0 — для padding
# idx2char = {i: c for c, i in char2idx.items()}
# 
# print(f"🔠 Алфавит ({len(vocab)} символов): {vocab}")

In [122]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MorseModel(nn.Module):
    def __init__(self, num_classes, input_features=64):  # input_features = n_mels
        super(MorseModel, self).__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )

        # после 2 MaxPool размер по time и freq делится на 4
        self.rnn = nn.GRU(input_size=(input_features // 4) * 64,
                          hidden_size=128,
                          num_layers=2,
                          batch_first=True,
                          bidirectional=True)

        self.classifier = nn.Linear(128 * 2, num_classes)  # bidirectional

    def forward(self, x):
        # x: (B, T, F) → (B, 1, T, F)
        x = x.unsqueeze(1)
        x = self.cnn(x)  # (B, C, T/4, F/4)

        B, C, T, F = x.size()
        x = x.permute(0, 2, 1, 3).contiguous()  # (B, T, C, F)
        x = x.view(B, T, C * F)  # (B, T, features)

        x, _ = self.rnn(x)  # (B, T, 256)
        x = self.classifier(x)  # (B, T, num_classes)

        return x.permute(1, 0, 2)  # CTC expects (T, B, C)


In [123]:
import torch
from torch.nn.utils.rnn import pad_sequence


def collate_fn(batch):
    specs, labels = zip(*batch)

    # specs: список тензоров формы (T, F)
    specs = [s for s in specs]  # (T, F) уже, не нужно transpose если так генерируешь

    # Паддинг по времени (T)
    specs_padded = pad_sequence(specs, batch_first=True)  # (B, T_max, F)

    # Метки
    label_lens = torch.tensor([len(t) for t in labels])
    targets = torch.cat([torch.tensor(t, dtype=torch.long) for t in labels])

    return specs_padded, targets, label_lens



In [124]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn_ctc)


In [125]:
class MorseModel(nn.Module):
    def __init__(self, num_classes, input_features=32):  # <= используем облегчённый спектр
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),  # было 32
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),  # было 64
            nn.ReLU(),
            nn.MaxPool2d((2, 2))
        )

        self.rnn = nn.GRU(
            input_size=(input_features // 4) * 32,  # было умножение на 64
            hidden_size=64,                         # было 128
            num_layers=1,                           # было 2
            batch_first=True,
            bidirectional=False                     # было True
        )

        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):  # x: (B, T, F)
        x = x.unsqueeze(1)           # → (B, 1, T, F)
        x = self.cnn(x)              # → (B, C, T/4, F/4)

        B, C, T, F = x.size()
        x = x.permute(0, 2, 1, 3)    # → (B, T, C, F)
        x = x.reshape(B, T, C * F)   # → (B, T, C*F)

        x, _ = self.rnn(x)           # → (B, T, H)
        x = self.classifier(x)       # → (B, T, C)
        return x.permute(1, 0, 2)    # → (T, B, C)


In [126]:
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm

# Настройки
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 8
epochs = 5

# DataLoader
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn_ctc)

# Модель
model = MorseModel(num_classes=len(char2idx), input_features=32).to(device)

# Оптимизатор и лосс
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.CTCLoss(blank=0, zero_infinity=True)

# Обучение
for epoch in range(epochs):
    model.train()
    total_loss = 0

    progress = tqdm(train_loader, desc=f"🚀 Эпоха {epoch+1}/{epochs}", leave=False)

    for i, (specs, targets_concat, target_lengths) in enumerate(progress):
        # specs: (B, F, T) → (B, T, F)
        specs = specs.permute(0, 2, 1).to(device)            # модель ожидает (B, T, F)
        targets_concat = targets_concat.to(device)
        target_lengths = target_lengths.to(device)

        optimizer.zero_grad()

        logits = model(specs)                                # (T, B, C)
        log_probs = F.log_softmax(logits, dim=2)

        input_lengths = torch.full(size=(logits.shape[1],), fill_value=logits.shape[0], dtype=torch.long).to(device)

        loss = criterion(log_probs, targets_concat, input_lengths, target_lengths)
        loss.backward()
        if epoch == 0 and i % 200 == 0:  # показываем каждые 200 шагов в 1-й эпохе
            decoded_preds = ctc_greedy_decode(log_probs)
    
            targets_split = torch.split(targets_concat, target_lengths.tolist())
    
            print("\n📊 Проверка CTC:")
            for true, pred in zip(targets_split[:3], decoded_preds[:3]):
                print("🟩 Истинно :", decode_indices(true.tolist()))
                print("🟦 Предсказ:", decode_indices(pred))
                print("---")
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    avg = total_loss / len(train_loader)
    print(f"✅ Эпоха {epoch+1}/{epochs} завершена | Средний лосс: {avg:.4f}")

🚀 Эпоха 1/5:   0%|          | 2/3750 [00:00<09:53,  6.32it/s, loss=16.8]


📊 Проверка CTC:
🟩 Истинно : НИ2ВИЩВВ7П
🟦 Предсказ: ЖЬЫЬЫЬЫЬЫЬЫЬЫЬЫЬЫЬ
---
🟩 Истинно : ЧГЖСЬЫЫД
🟦 Предсказ: В#Ь
---
🟩 Истинно : ТБО5ТЙХПЕВИЕЕ
🟦 Предсказ: ЪЫ4Ы4#4#4#4#4#Ы#4#
---


KeyboardInterrupt: 

In [100]:
def ctc_greedy_decode(log_probs, blank=0):
    # log_probs: (T, B, C) → выберем argmax по классам
    pred = torch.argmax(log_probs, dim=2).transpose(0, 1)  # (B, T)
    decoded = []

    for sequence in pred:
        prev = blank
        output = []
        for p in sequence:
            p = p.item()
            if p != prev and p != blank:
                output.append(p)
            prev = p
        decoded.append(output)
    
    return decoded  # список списков индексов


In [98]:
def decode_indices(indices):
    return ''.join([idx2char[i] for i in indices])


In [108]:
train_loader = DataLoader(
    dataset, batch_size=8,
    shuffle=True, collate_fn=collate_fn_ctc
)


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from dataset import MorseAudioDataset
import os
import pandas as pd
import numpy as np

# -----------------------------
# 1) Датасет
# -----------------------------

# -----------------------------
# 2) Функция для преобразования аудио -> Mel-спектрограмма
# -----------------------------
def audio_to_melspectrogram(y, sr=16000, n_mels=64, n_fft=1024, hop_length=512):
    # Получаем Mel-спектрограмму
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                       hop_length=hop_length, n_mels=n_mels)
    # логарифмируем (лог-Mel), чтобы было ближе к тому, как обрабатывается речь
    log_S = librosa.power_to_db(S, ref=np.max)
    # Нормируем к 0..1, например
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)
    return log_S_norm

# -----------------------------
# 3) Простая CNN-модель
# -----------------------------
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=3):
        """
        num_classes - у вас может быть 3 (dot, dash, silence),
                      или больше, если нужно.
        """
        super(SimpleCNN, self).__init__()
        # Предположим, что вход: [batch, 1, n_mels, time_frames]
        self.conv1 = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2)
        )
        # Можно добавить ещё один блок, если нужно
        self.fc = nn.Sequential(
            nn.Linear(32* (8) * (8), 64),  # числа 8x8 условные, зависят от входных размеров
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        # x.shape: [batch, 32, H, W]
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# -----------------------------
# 4) Собираем всё вместе
# -----------------------------
from tqdm import tqdm  # добавляем в начало файла

def train_loop(model, train_loader, val_loader, num_epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0

        print(f"\nEpoch {epoch+1}/{num_epochs}")
        train_iterator = tqdm(train_loader, desc="Training", leave=False)

        for batch_features, batch_labels in train_iterator:
            batch_features = batch_features.unsqueeze(1).to(device)
            batch_labels = batch_labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_features)
            loss = criterion(outputs, batch_labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            train_iterator.set_postfix(loss=loss.item())

        avg_loss = total_loss / len(train_loader)

        # Валидация
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        val_iterator = tqdm(val_loader, desc="Validation", leave=False)
        with torch.no_grad():
            for val_features, val_labels in val_iterator:
                val_features = val_features.unsqueeze(1).to(device)
                val_labels = val_labels.to(device)
                
                val_outputs = model(val_features)
                loss = criterion(val_outputs, val_labels)
                val_loss += loss.item()

                _, predicted = torch.max(val_outputs, 1)
                correct += (predicted == val_labels).sum().item()
                total += val_labels.size(0)
                val_iterator.set_postfix(loss=loss.item())

        val_avg_loss = val_loss / len(val_loader)
        accuracy = 100.0 * correct / total
        
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_loss:.4f} | "
              f"Val Loss: {val_avg_loss:.4f} | "
              f"Val Acc: {accuracy:.2f}%")


# -----------------------------
# 5) Пример использования
# -----------------------------
if __name__ == "__main__":
    # Допустим, у нас есть списки путей и меток
    # загружаем CSV
    train_df = pd.read_csv('train.csv')
    
    # делим на обучающую и валидационную выборки
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)
    
    # пути к аудиофайлам
    train_paths = [f'morse_dataset/{i}' for i in train_df['id']]
    train_labels = [i for i in train_df['message']]
    
    val_paths = [f'morse_dataset/{i}' for i in val_df['id']]
    val_labels = [i for i in val_df['message']]

    # Создадим датасеты
    # объяви функцию заранее
    def mel_transform(x):
        return audio_to_melspectrogram(x, sr=16000)
    
    # и передай её
    train_dataset = MorseAudioDataset(train_paths, train_labels, sr=16000,
                                      transform=mel_transform)
    
    val_dataset = MorseAudioDataset(val_paths, val_labels, sr=16000,
                                    transform=mel_transform)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)

    # Модель
    model = SimpleCNN(num_classes=3)  # или нужное число классов

    # Обучение
    train_loop(model, train_loader, val_loader, num_epochs=10, lr=1e-3)



Epoch 1/10


AttributeError: 'tuple' object has no attribute 'to'

In [ ]:
loss_fn = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import librosa
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random

############################################
# 1) Алфавит
############################################
alphabet = {
    '<pad>': 0,
    ' ': 1,
    '#': 2,
    '0': 3,
    '1': 4,
    '2': 5,
    '3': 6,
    '4': 7,
    '5': 8,
    '6': 9,
    '7': 10,
    '8': 11,
    '9': 12,
    'А': 13,
    'Б': 14,
    'В': 15,
    'Г': 16,
    'Д': 17,
    'Е': 18,
    'Ж': 19,
    'З': 20,
    'И': 21,
    'Й': 22,
    'К': 23,
    'Л': 24,
    'М': 25,
    'Н': 26,
    'О': 27,
    'П': 28,
    'Р': 29,
    'С': 30,
    'Т': 31,
    'У': 32,
    'Ф': 33,
    'Х': 34,
    'Ц': 35,
    'Ч': 36,
    'Ш': 37,
    'Щ': 38,
    'Ъ': 39,
    'Ы': 40,
    'Ь': 41,
    'Э': 42,
    'Ю': 43,
    'Я': 44
}
num_classes = len(alphabet)  # 45 (0 = blank)

############################################
# 2) encode_label
############################################
def encode_label(text: str, alpha: dict) -> list:
    indices = []
    for ch in text:
        if ch in alpha:
            indices.append(alpha[ch])
    return indices

############################################
# 3) Audio augment (до Mel)
############################################
def augment_waveform(y, sr=16000, noise_factor=0.005, time_stretch_range=(0.9, 1.1)):
    """
    Добавим шум и сделаем небольшой time-stretch для имитации разных условий.
    """
    # 1) Add random noise
    if random.random() < 0.5:
        # шум = noise_factor * random.randn
        noise = np.random.randn(len(y)) * noise_factor
        y = y + noise.astype(y.dtype)

    # 2) Time stretch
    if random.random() < 0.5:
        rate = random.uniform(*time_stretch_range)  # из (0.9..1.1) к примеру
        # librosa.effects.time_stretch
        # Если растянуть/сжать слишком сильно, может не хватить данных, обработаем try/except
        try:
            y = librosa.effects.time_stretch(y, rate=rate)
        except:
            pass

    return y

############################################
# 4) SpecAugment (для Mel)
############################################
def spec_augment(mel_spec, max_freq_mask=10, max_time_mask=20):
    """
    Простая реализация SpecAugment:
    - маскируем случайную полосу по частоте (freq)
    - маскируем случайную полосу по времени
    mel_spec: numpy array [n_mels, time]
    """
    # 1) Freq mask
    n_mels, n_time = mel_spec.shape
    if random.random() < 0.5:
        freq_mask_size = random.randint(1, max_freq_mask)
        f0 = random.randint(0, n_mels - freq_mask_size)
        mel_spec[f0:f0+freq_mask_size, :] = 0.0

    # 2) Time mask
    if random.random() < 0.5:
        time_mask_size = random.randint(1, max_time_mask)
        t0 = random.randint(0, n_time - time_mask_size)
        mel_spec[:, t0:t0+time_mask_size] = 0.0

    return mel_spec

############################################
# 5) Функция для Melspectrogram
#    (теперь с возможностью включить SpecAugment)
############################################
def audio_to_melspectrogram(y, sr=16000, n_mels=64, n_fft=1024, hop_length=512, do_specaug=False):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, 
                                       hop_length=hop_length, n_mels=n_mels)
    log_S = librosa.power_to_db(S, ref=np.max)
    # нормализуем
    log_S_norm = (log_S - log_S.min()) / (log_S.max() - log_S.min() + 1e-6)

    if do_specaug:
        # Применяем SpecAugment с некоторой вероятностью
        if random.random() < 0.7:
            log_S_norm = spec_augment(log_S_norm, max_freq_mask=10, max_time_mask=20)

    return log_S_norm  # [n_mels, time]

############################################
# 6) Dataset для CTC
############################################
class MorseAudioCTCDataset(Dataset):
    def __init__(self, df, sr=16000, transform=None, augment=False):
        """
        df: DataFrame c колонками ['id', 'message']
        sr: частота дискр
        transform: callable( y ) -> mel_spec
        augment: если True, включаем random аугментацию
        """
        self.df = df.reset_index(drop=True)
        self.sr = sr
        self.transform = transform
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.loc[idx]
        audio_path = "morse_dataset/" + row['id']  
        y, _ = librosa.load(audio_path, sr=self.sr)

        # If augment=True, применим augment_waveform
        if self.augment:
            y = augment_waveform(y, sr=self.sr)

        # Преобразуем в Mel
        if self.transform:
            mel = self.transform(y, sr=self.sr)
        else:
            # fallback
            mel = y

        # в тензор
        mel_tensor = torch.tensor(mel, dtype=torch.float)

        # label
        text_label = row['message']
        label_indices = encode_label(text_label, alphabet)
        label_tensor = torch.tensor(label_indices, dtype=torch.long)

        time_dim = mel_tensor.shape[1]

        return mel_tensor, label_tensor, time_dim

############################################
# 7) Collate_fn
############################################
def ctc_collate_fn(batch, pool_time_factor=4):
    mel_list = []
    label_list = []
    target_lengths_list = []
    raw_time_list = []

    for (mel, label, time_dim) in batch:
        mel_list.append(mel)
        label_list.append(label)
        target_lengths_list.append(len(label))
        raw_time_list.append(time_dim)

    max_time = max(m.shape[1] for m in mel_list)
    padded_mels = []
    for mel in mel_list:
        t = mel.shape[1]
        if t < max_time:
            pad_amount = max_time - t
            mel = F.pad(mel, (0, pad_amount), value=0.0)
        mel = mel.unsqueeze(0)  # => [1, n_mels, max_time]
        padded_mels.append(mel)

    audio_batch = torch.stack(padded_mels, dim=0) # [B, 1, n_mels, max_time]
    labels_concat = torch.cat(label_list, dim=0)

    input_lengths = []
    for tdim in raw_time_list:
        input_lengths.append(tdim // pool_time_factor)

    input_lengths = torch.tensor(input_lengths, dtype=torch.long)
    target_lengths = torch.tensor(target_lengths_list, dtype=torch.long)

    return audio_batch, labels_concat, input_lengths, target_lengths

############################################
# 8) Увеличенная модель
############################################
class EnhancedCTCModel(nn.Module):
    """
    - 2 Conv blocks
    - 2-layer BiLSTM
    - dropout
    - hidden_size=256
    """
    def __init__(self, num_classes=45, in_channels=1, n_mels=64,
                 hidden_size=256, lstm_layers=2, dropout=0.3):
        super(EnhancedCTCModel, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2))
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2,2))
        )
        # => freq/time /4, channels=32 => input_size=32*(n_mels//4)
        self.lstm = nn.LSTM(
            input_size=32 * (n_mels//4),
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            dropout=(dropout if lstm_layers>1 else 0.0),
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size*2, num_classes)

    def forward(self, x):
        """
        x: [B, 1, n_mels=64, time]
        -> [time//4, B, num_classes]
        """
        x = self.conv1(x)
        x = self.conv2(x)
        b,c,f,t = x.shape
        x = x.view(b, c*f, t)
        x = x.permute(2,0,1)  # [T, B, feature]
        lstm_out, _ = self.lstm(x) # [T, B, hidden*2]
        logits = self.fc(lstm_out) # [T, B, num_classes]
        return logits

############################################
# 9) Цикл обучения
############################################
def train_ctc_loop(model, train_loader, val_loader, num_epochs=10, lr=1e-3):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    ctc_loss = nn.CTCLoss(blank=0, reduction='mean', zero_infinity=True)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        
        # ---------- TRAIN ----------
        model.train()
        train_loss_sum = 0.0
        train_bar = tqdm(train_loader, desc="Train", leave=False)

        for audio_batch, labels_concat, input_lengths, target_lengths in train_bar:
            audio_batch = audio_batch.to(device)
            labels_concat = labels_concat.to(device)
            input_lengths = input_lengths.to(device)
            target_lengths = target_lengths.to(device)

            optimizer.zero_grad()
            logits = model(audio_batch)          
            log_probs = F.log_softmax(logits, dim=2) 
            
            loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = train_loss_sum / len(train_loader)

        # ---------- VALID ----------
        model.eval()
        val_loss_sum = 0.0
        val_bar = tqdm(val_loader, desc="Val", leave=False)
        with torch.no_grad():
            for audio_batch, labels_concat, input_lengths, target_lengths in val_bar:
                audio_batch = audio_batch.to(device)
                labels_concat = labels_concat.to(device)
                input_lengths = input_lengths.to(device)
                target_lengths = target_lengths.to(device)

                logits = model(audio_batch)
                log_probs = F.log_softmax(logits, dim=2)
                
                loss = ctc_loss(log_probs, labels_concat, input_lengths, target_lengths)
                val_loss_sum += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss_sum / len(val_loader)
        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


############################################
# 10) Main
############################################
if __name__ == "__main__":
    df = pd.read_csv("train.csv")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

    # 1) Создаём датасет c аугментацией для train, без аугментации для val
    #    + SpecAug на train
    def transform_train(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=True)

    def transform_val(y, sr=16000):
        return audio_to_melspectrogram(y, sr=sr, do_specaug=False)

    train_dataset = MorseAudioCTCDataset(
        train_df, 
        sr=16000, 
        transform=transform_train, 
        augment=True # включает augment_waveform
    )
    val_dataset = MorseAudioCTCDataset(
        val_df, 
        sr=16000, 
        transform=transform_val, 
        augment=False
    )

    # 2) DataLoader
    train_loader = DataLoader(
        train_dataset, 
        batch_size=4, 
        shuffle=True, 
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=4, 
        shuffle=False,
        num_workers=0,
        collate_fn=lambda b: ctc_collate_fn(b, pool_time_factor=4)
    )

    # 3) Модель
    model = EnhancedCTCModel(
        num_classes=45, 
        in_channels=1, 
        n_mels=64, 
        hidden_size=256, 
        lstm_layers=2, 
        dropout=0.3
    )

    # 4) Быстрая проверка на одном батче
    audio_batch, labels_concat, input_lengths, target_lengths = next(iter(train_loader))
    print("audio_batch shape:", audio_batch.shape)
    print("labels_concat shape:", labels_concat.shape)
    print("input_lengths:", input_lengths)
    print("target_lengths:", target_lengths)

    # 5) Запускаем обучение
    train_ctc_loop(model, train_loader, val_loader, num_epochs=15, lr=1e-3)
